# 06_llm_prompt_design 

LLM prompt iteration on 20-post dev probe before full eval run. Keep cost under .

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np

root_dir = Path(os.getcwd()).parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
# Eigen scripts

from src.data import load_splits
from src.llm.prompts import build_prompt
from src.llm.openai_client import classify as classify_openai
from src.llm.openai_client import classify as classify_gpt
from src.llm.gemini_client import classify as classify_gemini

# Laad de API keys
load_dotenv()
print("API Keys geladen:", "GOOGLE_API_KEY" in os.environ)

In [ ]:
train_df, val_df, test_df = load_splits()

# Test op 20 posts
test_subset = train_df.sample(20, random_state=42)

# Haal de few-shot voorbeelden uit de rest van de training set
few_shot_examples = (
    train_df.drop(test_subset.index)
    .groupby("label_id")
    .apply(lambda g: g.sample(2, random_state=42))
    .reset_index(drop=True)[["text", "label"]]
    .to_dict("records")
)

print(f"Subset van {len(test_subset)} posts klaar voor test.")

In [ ]:
# Check de prompt voor de eerste post uit je subset
sample_post = test_subset.iloc[0]['text']
prompt_check = build_prompt(variant="few_shot", post=sample_post, few_shot_examples=few_shot_examples)

print("--- VOORBEELD PROMPT ---")
print(prompt_check)

CHATGPT TEST

In [ ]:
# (Zorg dat je 'build_prompt' ook importeert als dat in een ander bestand staat)

results = []

for i, (idx, row) in enumerate(test_subset.iterrows()):
    # 1. Bouw de prompt (Chain of Thought variant zoals gevraagd)
    prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
    # 2. Roep de OpenAI client aan met de config voEor GPT-5
    res = classify_openai(prompt, config={"model": "gpt-5", "temperature": 0.0, "max_tokens": 800}
    )
    
    res['true_label'] = row['label']
    res['post_id'] = idx
    
    results.append(res)
    
    print(f"Post {i+1}: GPT-5 koos '{res['label']}' (Echt: {res['true_label']})")

# Maak de uiteindelijke DataFrame
test_results_df = pd.DataFrame(results)

Post 1 resultaat:


NameError: name 'json' is not defined

GEMINI TEST

In [ ]:
results = []

for i, (idx, row) in enumerate(test_subset.iterrows()):
    # We testen Chain of Thought omdat die het meest complex is
    prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
    
    # Roep de client aan
    res = classify_gemini(prompt, config={"model": "gemini-3.5-flash", "temperature": 0.0, "max_tokens":1600})
    
    # Sla resultaat op
    res['true_label'] = row['label']
    results.append(res)
    print(res['label'])
    # print(f"Post {i+1}/20: Model koos '{res['label']}' (Echt: {res['true_label']})")

test_results_df = pd.DataFrame(results)

In [ ]:
avg_cost = test_results_df['cost_usd'].mean()
total_estimated = avg_cost * 1040 * 3 * 2 # 1040 posts, 3 varianten, 2 modellen

print(f"Gemiddelde kosten per call: ${avg_cost:.4f}")
print(f"Geschatte totale kosten: ${total_estimated:.2f}")

# Laat de eerste 5 resultaten zien inclusief de reasoning
test_results_df[['true_label', 'label', 'reasoning', 'cost_usd']].head()